In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import joblib
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patches as patches
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

from utils.plotting import df_to_gridded_da
import config
from nn_utils.dataset import load_dataset
from tuning_studies.modelconfig import ablation_study

In [ ]:
# Model testing settings
test_dir_names = ["output/testing/mastnet/",
                  "output/testing/mean/",
                  "output/testing/missforest/",
                  "output/testing/knn/",
                  "output/testing/mlp/",
                  "output/testing/remasker/"
                  ]
imputer_name_map = {"missforest": "MissForest",
                    "missforest_": "MissForest",
                    "mastnet": "MaSTNeT",
                    "mastnet_": "MaSTNeT",
                    "remasker": "ReMasker",
                    "remasker_": "ReMasker",
                    "mean": "Mean",
                    "mean_": "Mean",
                    "knn": "KNN",
                    "knn_": "KNN",
                    "mlp": "FFN",
                    "mlp_": "FFN",
                    }

# Ablation settings
ablation_base_dir = Path("D:/phd/output/tuning/mastnet/architecture_ablation/")
per_sample_num_map = {0.3: 1, 0.4: 2, 0.5: 3, 0.7: 4, 0.9: 5}
baseline_cfg = ablation_study["exp0"]["config"]
attention_name_map = {"space_time_attention": "Space-time attention",
                      "mha": "MHA",
                      "autoencoder": "Autoencoder",
                      "encoder_decoder": "Autoencoder",
                      "space_time_depth_attention": "Space-time-depth attention",
                      "transformer_encoder_layer": "Transformer encoder"
                      }

In [ ]:
# Load raw data
df = load_dataset()

# Load test idxs
test_idx = json.load(open(f"{config.output_dir_splits}/test_train_split.json", "r"))["test_idx"]

In [ ]:
def get_attr(res, attr_name):
    if attr_name in res["hyps"]["model"]["cfg"].keys():
        return res["hyps"]["model"]["cfg"][attr_name]
    else:
        return getattr(baseline_cfg, attr_name)

def load_experiment(dir_path, model_name="mastnet", woa=True, cmip=True):
    df_woa, df_cmip = [], []

    file_path = Path(dir_path) / f"model{model_name}_splitfinal_trial0.json"
    if not Path(file_path).is_file():
        print(f"No such file: {file_path}")
        return None, None, None
    res = json.load(open(file_path, "r"))
    metrics_field = "test_metrics" if res["model_framework"] == "pytorch" else "metrics_all"
    metrics = pd.DataFrame([res[metrics_field]["Global"]])
    metrics["exp_id"] = exp_id

    # Add mask ratio info (if available)
    if "model" in res["hyps"].keys():
        if "cfg" in res["hyps"]["model"].keys():
            metrics["mask_ratio"] = get_attr(res, "mask_ratio")  # res["hyps"]["model"]["cfg"]["mask_ratio"]
            metrics["transect_mask_orientation"] = get_attr(res, "transect_mask_orientation")  # res["hyps"]["model"]["cfg"]["transect_mask_orientation"]
            metrics["sphere_mask_radius"] = get_attr(res, "sphere_mask_radius")  # res["hyps"]["model"]["cfg"]["sphere_mask_radius"]
            metrics["masking_strategies"] = str(get_attr(res, "masking_strategies"))
            metrics["transect_mask_width"] = str(get_attr(res, "transect_mask_width"))

    # WOA data (scaled)
    if woa:
        w = pd.read_csv(dir_path / f"{model_name}_woa_diff_summary_scaled.csv")
        w["exp_id"] = exp_id
        df_woa.append(w)
        metrics["WOA_diff"] = w["MAE"].mean()
        metrics["WOA_std"] = w["std"].mean()

    # CMIP data (scaled)
    if cmip:
        c = pd.read_csv(dir_path / f"{model_name}_cmip_diff_summary_scaled.csv")
        c["exp_id"] = exp_id
        df_cmip.append(c)
        metrics["CMIP_diff"] = c["MAE"].mean()
        metrics["CMIP_std"] = c["std"].mean()

    # if "attention_type" in metrics.columns:
    #     metrics["attention_type"] = metrics["attention_type"].map(attention_name_map)

    df_woa = None if len(df_woa) == 0 else pd.concat(df_woa)
    df_cmip = None if len(df_cmip) == 0 else pd.concat(df_cmip)

    return metrics, df_woa, df_cmip

In [ ]:
def extract_mean_miss_ratio(exp_id="exp113"):
    dir_path = ablation_base_dir / exp_id
    history = json.load(open(dir_path / "mastnet_history.json", "r"))

    avg_miss_ratio = np.nan
    if "train_miss_ratio" in history.keys():
        miss_ratios = history["train_miss_ratio"]
        avg_miss_ratio = np.mean(list(miss_ratios.values()))
    else:
        print(f"No missiningess ratios found for {exp_id}")

    return avg_miss_ratio

In [ ]:
def plot_depth_maps(
    res_dict,
    models_to_plot,
    imputer_name_map,
    depths,
    parameter,
    nrows,
    ncols,
    plot_config=None,
    mode_list=["mean", "std"],
    output_path=None,
    prefix="reconstruction",
    dpi=300,
):
    cmap = "inferno"

    for mode in mode_list:
        fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 3.2, nrows * 2), subplot_kw={"projection": ccrs.PlateCarree()})
        data_dict = {}
        vmin, vmax = np.inf, -np.inf

        # Prepare data
        for model in models_to_plot:
            temp = res_dict[model][mode]
            da = df_to_gridded_da(temp, value_col=parameter)
            da = da.transpose("depth", "lat", "lon")

            name = imputer_name_map[model]
            data_dict[name] = da

            vmin = min(vmin, float(da.min()))
            vmax = max(vmax, float(da.max()))

        pcm_list = []

        # Plot
        for col, model in enumerate(models_to_plot):
            model_name = imputer_name_map[model]
            data = data_dict[model_name]

            for row, d in enumerate(depths):
                ax = axs[row, col] if nrows > 1 else axs[col]
                ax.set_extent([float(data.lon.min()), float(data.lon.max()), float(data.lat.min()), float(data.lat.max())], crs=ccrs.PlateCarree())
                ax.set_aspect("auto")

                ax.add_feature(cfeature.LAND, facecolor="lightgrey", zorder=0)
                ax.coastlines(linewidth=0.5)
                ax.add_feature(cfeature.BORDERS, linewidth=0.3)

                da2d = data.sel(depth=d)

                cfg = plot_config.get(mode, {}) if plot_config else {}

                cmap_use = cfg.get("cmap", cmap)
                norm_use = cfg.get("norm", None)
                vmin_use = cfg.get("vmin", vmin)
                vmax_use = cfg.get("vmax", vmax)

                if norm_use is not None:
                    pcm = ax.pcolormesh(data.lon, data.lat, da2d, cmap=cmap_use, norm=norm_use, transform=ccrs.PlateCarree())
                else:
                    pcm = ax.pcolormesh(data.lon, data.lat, da2d, cmap=cmap_use, vmin=vmin_use, vmax=vmax_use, transform=ccrs.PlateCarree())
                pcm_list.append(pcm)

                # Gridlines
                gl = ax.gridlines(draw_labels=True, linewidth=0.3, linestyle="--", alpha=0.5)

                # Disable top/right
                gl.top_labels = False
                gl.right_labels = False

                # Longitude formatting W/E
                gl.xformatter = LONGITUDE_FORMATTER
                gl.yformatter = LATITUDE_FORMATTER

                gl.xlabel_style = {"size": 8}
                gl.ylabel_style = {"size": 8}

                # Label control
                # Left column: Show latitude labels
                if col == 0:
                    gl.left_labels = True
                    ax.set_ylabel(f"{d} m", labelpad=25)
                    ax.set_yticks([])
                    ax.yaxis.set_visible(True)
                else:
                    gl.left_labels = False

                # Bottom row: Show longitude labels
                if row == nrows - 1:
                    gl.bottom_labels = True
                else:
                    gl.bottom_labels = False

                # Title
                if row == 0:
                    ax.set_title(model_name)

        # Shared colour bar
        cbar = fig.colorbar(pcm_list[0], ax=axs, orientation="vertical", shrink=0.9, aspect=30, pad=0.02)
        cbar.set_label(config.parameter_name_unit_map[parameter])

        if output_path is not None:
            plt.savefig(f"{output_path}/{prefix}_depth_maps_{parameter}_{mode}.png", dpi=dpi, bbox_inches="tight")
        plt.show()


# Missingness plot

In [ ]:
# Compute missingness per row
temp = df.copy()
temp["miss"] = temp[config.parameters].isna().sum(axis=1) / len(config.parameters)

# Compute mean missingness over time
df_miss_avg = temp.groupby(["LATITUDE", "LONGITUDE", "LEV_M"])["miss"].mean().reset_index()

In [ ]:
# def plot_missingness_panels(
#     df,
#     depth_levels,
#     cmap="RdYlGn_r",
#     ncols=3,
#     vmin=0,
#     vmax=1,
#     save_as=None
# ):
#     n = len(depth_levels)
#     nrows = int(np.ceil(n / ncols))
#
#     fig, axs = plt.subplots(
#         nrows=nrows,
#         ncols=ncols,
#         figsize=(ncols * 3.5, nrows * 3),
#         subplot_kw={"projection": ccrs.PlateCarree()},
#         constrained_layout=True,
#     )
#     axs = axs.flatten()
#
#     pcm = None
#
#     for i, d in enumerate(depth_levels):
#         ax = axs[i]
#
#         sub = df[df["LEV_M"] == d]
#
#         grid = sub.pivot_table(
#             index="LATITUDE",
#             columns="LONGITUDE",
#             values="miss"
#         )
#
#         lats = grid.index.values
#         lons = grid.columns.values
#         data = grid.values
#
#         lon_min, lon_max = df["LONGITUDE"].min(), df["LONGITUDE"].max()
#         lat_min, lat_max = df["LATITUDE"].min(), df["LATITUDE"].max()
#         ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
#         #ax.set_extent([lons.min(), lons.max(), lats.min(), lats.max()])
#
#         ax.add_feature(cfeature.LAND, facecolor="lightgrey", zorder=0)
#         ax.coastlines(linewidth=0.5)
#
#         gl = ax.gridlines(draw_labels=True, linewidth=0.3, linestyle="--")#, alpha=0.9)
#         gl.top_labels = False
#         gl.right_labels = False
#         gl.xlabel_style = {"size": 8}
#         gl.ylabel_style = {"size": 8}
#
#         pcm = ax.pcolormesh(
#             lons,
#             lats,
#             data,
#             cmap=cmap,
#             vmin=vmin,
#             vmax=vmax,
#             shading="auto",
#             transform=ccrs.PlateCarree(),
#         )
#
#         ax.set_title(f"{int(d)} m", fontsize=9)
#
#     # remove empty panels
#     for j in range(i + 1, len(axs)):
#         axs[j].axis("off")
#
#     # shared colorbar
#     cbar = fig.colorbar(pcm, ax=axs, orientation="vertical", pad=0.02, shrink=0.79)
#     cbar.set_label("Missingness")
#
#     if save_as:
#         plt.savefig(save_as, dpi=150, facecolor="white")
#
#     plt.show()

In [ ]:
def plot_missingness_panels(
    df,
    depth_levels,
    cmap="RdYlGn_r",
    ncols=3,
    vmin=0,
    vmax=1,
    save_as=None
):
    n = len(depth_levels)
    nrows = int(np.ceil(n / ncols))

    fig, axs = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(ncols * 3.6, nrows * 3.2),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True,
    )

    axs = np.atleast_1d(axs).flatten()

    pcm = None

    lon_min, lon_max = df["LONGITUDE"].min(), df["LONGITUDE"].max()
    lat_min, lat_max = df["LATITUDE"].min(), df["LATITUDE"].max()

    for i, d in enumerate(depth_levels):
        ax = axs[i]

        sub = df[df["LEV_M"] == d]

        grid = sub.pivot_table(
            index="LATITUDE",
            columns="LONGITUDE",
            values="miss"
        )

        lats = grid.index.values
        lons = grid.columns.values
        data = grid.values

        ax.set_extent(
            [lon_min, lon_max, lat_min, lat_max],
            crs=ccrs.PlateCarree()
        )

        # land + coastlines (light, publication style)
        ax.add_feature(cfeature.LAND, facecolor="0.92", zorder=0)
        ax.coastlines(linewidth=0.6, color="0.3")

        # ---- gridlines with controlled labeling ----
        row = i // ncols
        col = i % ncols

        gl = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=0.4,
            linestyle="--",
            # alpha=0.1,
            color="gray"
        )

        # turn everything off first
        gl.top_labels = False
        gl.right_labels = False
        gl.left_labels = False
        gl.bottom_labels = False

        # enable only outer labels
        if col == 0:
            gl.left_labels = True
            gl.ylabel_style = {"size": 8}

        if row == nrows - 1:
            gl.bottom_labels = True
            gl.xlabel_style = {"size": 8}

        # plot data
        pcm = ax.pcolormesh(
            lons,
            lats,
            data,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            shading="auto",
            transform=ccrs.PlateCarree(),
        )

        ax.set_title(f"{int(d)} m", fontsize=9)

    # turn off empty panels
    for j in range(i + 1, len(axs)):
        axs[j].axis("off")

    # shared colorbar
    cbar = fig.colorbar(
        pcm,
        ax=axs,
        orientation="vertical",
        pad=0.02,
        shrink=0.7
    )
    cbar.set_label("Missingness", fontsize=10)

    if save_as:
        plt.savefig(save_as, dpi=300, bbox_inches="tight", facecolor="white")

    plt.show()

In [ ]:
plot_missingness_panels(
    df_miss_avg,
    depth_levels=[0, 500, 3000],
    cmap="RdYlGn_r",
    ncols=3,
    vmin=0,
    vmax=1,
    save_as="output/plots/data_exploration/missingness_mean.png"
)

# Error plots

In [ ]:
# Load statistics
summary = []
for dir_name in test_dir_names:

    file_name = Path(dir_name)
    model_name = file_name.parts[2]
    exp_id = next((part for part in file_name.parts if re.fullmatch(r"exp\d+", part)), "")

    df_temp, df_woa, df_cmip = load_experiment(dir_path=file_name, model_name=model_name, woa=True, cmip=True)

    # df_temp = pd.read_csv(file_name)
    df_temp["model_name"] = model_name
    df_temp["exp_id"] = exp_id
    df_temp["desc"] = model_name + "_" + exp_id

    summary.append(df_temp)

summary = pd.concat(summary)


In [ ]:
# Define colour palette for models
models = sorted(summary["desc"].unique())
palette = dict(zip(models, sns.color_palette("tab10", n_colors=len(models))))

In [ ]:
# RMSE (on test set)
temp = summary.sort_values("RMSE")
plt.figure(figsize=[10, 6])
sns.barplot(y="RMSE", x="desc", hue="desc", data=temp, palette=palette)
plt.xlabel("")
plt.ylabel("RMSE")
plt.show()

In [ ]:
# WOA diff (time average 1950-2020)
temp = summary.sort_values("WOA_diff")
plt.figure(figsize=[10, 6])
sns.barplot(y="WOA_diff", x="desc", hue="desc", data=temp, palette=palette)
plt.xlabel("")
plt.ylabel("WOA_diff")
plt.show()

In [ ]:
# CMIP diff (per 20-years, 1852-2020)
temp = summary.sort_values("CMIP_diff")
plt.figure(figsize=[10, 6])
sns.barplot(y="CMIP_diff", x="desc", hue="desc", data=temp, palette=palette, legend=False)
sns.scatterplot(y="CMIP_std", x="desc", hue="desc", data=temp, palette=palette, legend=False)
plt.axhline(y=temp[temp["desc"] == "mastnet_"]["CMIP_std"].values[0], color="red", linestyle="--", label="mastnet")
plt.axhline(y=temp[temp["desc"] == "mastnet_"]["CMIP_diff"].values[0], color="red", linestyle="--", label="mastnet_mean")
plt.xlabel("")
plt.ylabel("CMIP_diff")
plt.show()

In [ ]:
# Boxplot
# Compute AE for each row and show as boxplot
res_list = []
for dir_name in test_dir_names:
    print(dir_name)
    model_name = Path(dir_name).parts[-1] if "mastnet" not in dir_name else "mastnet"
    df_true = pd.read_csv(Path(dir_name) / f"model{model_name}_y_true.csv")
    df_pred = pd.read_csv(Path(dir_name) / f"model{model_name}_y_pred.csv")

    if "Unnamed: 0" in df_true.columns:
        df_true = df_true.drop("Unnamed: 0", axis=1)

    if "Unnamed: 0" in df_pred.columns:
        df_pred = df_pred.drop("Unnamed: 0", axis=1)

    # Only not-nan entries on test data
    df_true_test = df_true.iloc[test_idx]
    df_pred_test = df_pred.iloc[test_idx]

    result = np.abs(df_true_test - df_pred_test)
    result = result[df_true_test.notna()]

    df_long = result.melt(id_vars=[], var_name="Parameter", value_name="Absolute Error")
    df_long["Model"] = model_name

    res_list.append(df_long)

df_ae = pd.concat(res_list)

# Sort with model median
order = df_ae.groupby("Model")["Absolute Error"].median().sort_values().index
df_ae["Model"] = pd.Categorical(df_ae["Model"], categories=order, ordered=True)
df_ae["Model"] = df_ae["Model"].map(imputer_name_map)

df_ae = df_ae.dropna().reset_index(drop=True)

In [ ]:
plt.figure(figsize=(6, 6))
hue_order = [imputer_name_map[o] for o in order]
ax = sns.ecdfplot(data=df_ae, x="Absolute Error", hue="Model", hue_order=hue_order)
sns.move_legend(ax, loc="lower center", ncol=3)

ax.set_xlabel("Absolute Error")
ax.set_ylabel("Proportion ≤ x")
ax.set_ylim(0, 1.01)

# Insert zoom plot
axins = inset_axes(ax, width="45%", height="45%", loc="center right")
sns.ecdfplot(data=df_ae, x="Absolute Error", hue="Model", hue_order=hue_order, ax=axins, legend=False)

# Zoom region
axins.set_xlim(0, 0.1)
axins.set_ylim(0.9, 1)
axins.set_title("Zoom", fontsize=9)
axins.set_xlabel("")
axins.set_ylabel("")

# Box around zoom region on original plot
rect = patches.Rectangle(xy=(0, 0.9), width=0.1, height=0.1, linewidth=1, edgecolor="gray", facecolor="none", linestyle="--", alpha=0.5)
ax.add_patch(rect)

# Get colors used by seaborn
medians = df_ae.groupby("Model", observed=True)["Absolute Error"].median()
palette = sns.color_palette(n_colors=len(hue_order))
color_map = dict(zip(hue_order, palette))

for i, model in enumerate(models):
    x = medians[imputer_name_map[model]]
    axins.vlines(x=x, ymin=0.9, ymax=0.905, linewidth=1, color=color_map[imputer_name_map[model]])

plt.tight_layout()
plt.savefig(f"{config.output_dir_plots}/test_evaluation/test_rmse_ecdf.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(8, 10))
sns.boxplot(data=df_ae, y="Absolute Error", x="Model")
plt.xlabel("")
plt.tight_layout()
plt.savefig(f"{config.output_dir_plots}/test_evaluation/test_rmse_boxplot.png", dpi=300)
plt.show()

# Reconstruction plots

(not imputation plots - maybe change?) @todo

In [ ]:
depths = [0, 200, 1000]  # epi meso bathy-pelagic

# Models to plot
models_to_plot = ["mlp_", "missforest_", "mastnet_", "remasker_", "knn_"]
ncols = len(models_to_plot)
nrows = len(depths)

In [ ]:
# Load predictions
res = {}
for dir_name in test_dir_names:
    print(dir_name)

    file_name = Path(dir_name)
    model_name = file_name.parts[2]
    exp_id = next((part for part in file_name.parts if re.fullmatch(r"exp\d+", part)), "")

    # Load prediction
    df_pred = pd.read_csv(file_name / f"model{model_name}_y_pred.csv")

    # Undo scaling
    scalers = joblib.load(file_name / "scalers.joblib")
    for k, v in scalers.items():
        if k in df_pred.columns:
            df_pred[k] = v.inverse_transform(df_pred[k].to_numpy().reshape(-1, 1))

    # Add coordinate information and clean up
    if "Unnamed: 0" in df_pred.columns:
        df_pred = df_pred.drop("Unnamed: 0", axis=1)
    df_pred[config.coordinates] = df[config.coordinates]

    # Time average and std
    df_time_avg = df_pred.drop("DATEANDTIME", axis=1).groupby(["LATITUDE", "LONGITUDE", "LEV_M"]).mean().reset_index()
    df_time_std = df_pred.drop("DATEANDTIME", axis=1).groupby(["LATITUDE", "LONGITUDE", "LEV_M"]).std().reset_index()

    # Filter for depths
    df_filtered_avg = df_time_avg[df_time_avg["LEV_M"].isin(depths)]
    df_filtered_std = df_time_std[df_time_std["LEV_M"].isin(depths)]

    res[model_name + "_" + exp_id] = {}
    res[model_name + "_" + exp_id]["mean"] = df_filtered_avg
    res[model_name + "_" + exp_id]["std"] = df_filtered_std

In [ ]:
plot_config = {
    "mean": {
        "cmap": "inferno",
        "norm": None,
    },
    "std": {
        "cmap": "viridis",
        "norm": None,
    }
}

for parameter in config.parameters:
    plot_depth_maps(
        res_dict=res,
        models_to_plot=models_to_plot,
        imputer_name_map=imputer_name_map,
        depths=depths,
        parameter=parameter,
        plot_config=plot_config,
        nrows=nrows,
        ncols=ncols,
        mode_list=["mean", "std"],
        output_path="output/plots/test_evaluation/reconstructions",
        prefix="reconstruction",
        dpi=300
    )

# WOA differences

In [ ]:
min_year = 1952

lat_min, lat_max = df.LATITUDE.min(), df.LATITUDE.max()
lon_min, lon_max = df.LONGITUDE.min(), df.LONGITUDE.max()
depths = np.sort(df.LEV_M.unique())
depths = np.append(depths, 5000)

# Load WOA data dict
woa_dict = {}
for param in config.parameters:
    woa_dict[param] = pd.read_parquet(f"../../data/woa/woa_{param}.parquet")

In [ ]:
def compute_woa_diffs(df_rec, df_true, woa_dict, scaler_dict=None):
    df_compares = []
    for parameter in config.parameters:
        temp_woa = woa_dict[parameter].copy()

        # Scaling
        if scaler_dict is not None:
            if parameter in scaler_dict.keys():
                temp_woa[parameter + "_woa"] = scaler_dict[parameter].transform(temp_woa[parameter].to_numpy().reshape(-1, 1))
        else:
            temp_woa[parameter + "_woa"] = temp_woa[parameter]

        # Rounding
        temp_woa["lat_round"] = temp_woa["LATITUDE"].round(1)
        temp_woa["lon_round"] = temp_woa["LONGITUDE"].round(1)

        # Only compare values that had been missing
        df_rec["was_missing"] = df_true[parameter].isna().values
        df_missing = df_rec[df_rec["was_missing"]].copy()
        df_missing = df_missing[df_missing["DATEANDTIME"] >= min_year]

        df_compare = df_missing[["LEV_M", "lat_round", "lon_round", parameter]].merge(temp_woa[["LEV_M", "lat_round", "lon_round", parameter + "_woa"]],
                                      on=["LEV_M", "lat_round", "lon_round"],
                                      #left_on=["DATEANDTIME", "lat_round", "lon_round", "LEV_M"],
                                      #right_on=["year_bins", "lat_round", "lon_round", "lev_bins"],
                                      how="inner")

        df_compare[parameter] = df_compare[parameter] - df_compare[parameter + "_woa"]
        df_compare = df_compare[[parameter, "lat_round", "lon_round", "LEV_M"]].rename(columns={"lat_round": "LATITUDE", "lon_round": "LONGITUDE"})
        df_compares.append(df_compare)

    # Assemble parameters into one df
    temp = df_compares[0]
    for t in df_compares[1:]:
        temp = temp.merge(t, on=["LATITUDE", "LONGITUDE", "LEV_M"], how="outer")
    temp_group = temp.groupby(["LATITUDE", "LONGITUDE", "LEV_M"])

    # Mean and std over time
    temp_mean = temp_group.mean().reset_index()
    temp_std = temp_group.std().reset_index()
    return {"mean": temp_mean, "std": temp_std}


In [ ]:
res_woa = {}
res_woa_scaled = {}

# Loop over models
for dir_name in test_dir_names:
    print(dir_name)

    model_out_dir = Path(dir_name)
    model_name = model_out_dir.parts[-1]
    exp_id = "" if model_out_dir.parts[-1] == model_name else model_out_dir.parts[-1]

    # Load data
    y_pred = pd.read_csv(model_out_dir / f"model{model_name}_y_pred.csv")
    y_true = pd.read_csv(model_out_dir / f"model{model_name}_y_true.csv")

    # Cleanup
    if "Unnamed: 0" in y_pred.columns:
        y_pred = y_pred.drop(columns="Unnamed: 0")

    # Inverse scaling
    y_pred_unscaled = y_pred.copy()
    scaler_dict = joblib.load(model_out_dir / "scalers.joblib")
    for k, v in scaler_dict.items():
        if k in y_pred_unscaled.columns:
            y_pred_unscaled[k] = v.inverse_transform(y_pred_unscaled[k].to_numpy().reshape(-1, 1))

    # Add coordinates
    y_pred[config.coordinates] = df[config.coordinates]
    df_rec = pd.DataFrame(y_pred)

    y_pred_unscaled[config.coordinates] = df[config.coordinates]
    df_rec_unscaled = pd.DataFrame(y_pred_unscaled)

    df_true = pd.DataFrame(y_true)
    df_true[config.coordinates] = df[config.coordinates]

    # Precompute rounded coords
    df_rec["lat_round"] = df_rec["LATITUDE"].round(1)
    df_rec["lon_round"] = df_rec["LONGITUDE"].round(1)

    df_rec_unscaled["lat_round"] = df_rec_unscaled["LATITUDE"].round(1)
    df_rec_unscaled["lon_round"] = df_rec_unscaled["LONGITUDE"].round(1)

    a = compute_woa_diffs(df_rec=df_rec_unscaled, df_true=df_true, woa_dict=woa_dict, scaler_dict=None)
    b = compute_woa_diffs(df_rec=df_rec, df_true=df_true, woa_dict=woa_dict, scaler_dict=scaler_dict)

    # Store
    res_woa[model_name] = a
    res_woa_scaled[model_name] = b

In [ ]:
# ECDF plot (scaled WOA diffs)
df_woa_diff = []
for model_name, temp_res in res_woa_scaled.items():
    temp = temp_res["mean"]
    temp["mean_diff"] = temp[config.parameters].abs().mean(axis=1)
    temp["std_diff"] = temp[config.parameters].abs().std(axis=1)
    temp["Model"] = model_name
    df_woa_diff.append(temp)

df_woa_diff = pd.concat(df_woa_diff).reset_index()
df_woa_diff["Model"] = df_woa_diff["Model"].map(imputer_name_map)

# Plot
plt.figure(figsize=(6, 6))
hue_order = [imputer_name_map[o] for o in order]
ax = sns.ecdfplot(data=df_woa_diff, x="mean_diff", hue="Model", hue_order=hue_order)
sns.move_legend(ax, loc="lower center", ncol=3)

ax.set_xlabel("|WOA - pred|")
ax.set_ylabel("Proportion ≤ x")
ax.set_ylim(0, 1.01)

# Insert zoom plot
s = 45
axins = inset_axes(ax, width=f"{s}%", height=f"{0.05/0.05 * s}%", loc="center right")
sns.ecdfplot(data=df_woa_diff, x="mean_diff", hue="Model", hue_order=hue_order, ax=axins, legend=False)

# Zoom region
axins.set_xlim(0.0, 0.1)
axins.set_ylim(0.9, 1.0)
axins.set_title("Zoom", fontsize=9)
axins.set_xlabel("")
axins.set_ylabel("")

# Box around zoom region on original plot
rect = patches.Rectangle(xy=(0.0, 0.9), width=0.1, height=0.1, linewidth=1, edgecolor="gray", facecolor="none", linestyle="--", alpha=0.5)
ax.add_patch(rect)

# Get colors used by seaborn
medians = df_woa_diff.groupby("Model", observed=True)["mean_diff"].median()
palette = sns.color_palette(n_colors=len(hue_order))
color_map = dict(zip(hue_order, palette))

for i, model in enumerate(models):
    x = medians[imputer_name_map[model]]
    axins.vlines(x=x, ymin=0.9, ymax=0.905, linewidth=1, color=color_map[imputer_name_map[model]])

plt.tight_layout()
plt.savefig(f"{config.output_dir_plots}/test_evaluation/woa_diff/woa_diff_ecdf.png", dpi=300)
plt.show()

In [ ]:
res_to_plot = {"scaled": res_woa_scaled, "unscaled": res_woa}

for scaling, temp_res in res_to_plot.items():
    for parameter in config.parameters:
        print(parameter)

        # Compute norm for cmap
        df_all_param = []
        for model in [m.rstrip("_") for m in models_to_plot]:
            df_model = temp_res[model]["mean"]
            df_all_param.append(df_model[parameter].dropna())
        df_all_param = pd.concat(df_all_param)

        absmax = np.max(np.abs(df_all_param))
        mean_norm = mcolors.TwoSlopeNorm(vmin=-absmax, vcenter=0, vmax=absmax)

        plot_config = {
            "mean": {
                "cmap": "seismic",
                "norm": mean_norm,
            },
            "std": {
                "cmap": "viridis",
                "norm": None,
            }
        }

        plot_depth_maps(
            res_dict=temp_res,
            models_to_plot=[m.rstrip("_") for m in models_to_plot],
            imputer_name_map=imputer_name_map,
            depths=[0, 200, 1000],
            parameter=parameter,
            plot_config=plot_config,
            nrows=3,
            ncols=len(models_to_plot),
            mode_list=["mean", "std"],
            output_path="output/plots/test_evaluation/woa_diff",
            prefix=f"woa_diff_{scaling}",
            dpi=300
        )

# CMIP differences

In [ ]:
# Load CMIP data dict
var_ids = ["thetao", "so", "o2", "no3", "si", "po4"]
min_year = 1852

cmip_dict = {}
for param in config.parameters:
    cmip_dict[param] = pd.read_parquet(f"../../data/cmip_data/cmip_{param}.parquet")

In [ ]:
def compute_cmip_diffs(df_rec, df_true, cmip_dict, scaler_dict=None):
    df_compares = []
    for var_id, parameter in zip(var_ids, config.parameters):
        temp_cmip = cmip_dict[parameter].copy()

        # Scaling
        if scaler_dict is not None:
            if parameter in scaler_dict.keys():
                temp_cmip[var_id] = scaler_dict[parameter].transform(temp_cmip[var_id].to_numpy().reshape(-1, 1))

        df_rec["lat_round"] = df_rec["LATITUDE"].round(1)
        df_rec["lon_round"] = df_rec["LONGITUDE"].round(1)
        df_rec["was_missing"] = df_true[parameter].isna().values

        # Only compare values that had been missing
        df_missing = df_rec[df_rec["was_missing"]].copy()
        df_missing = df_missing[df_missing["DATEANDTIME"] >= min_year]

        df_compare = df_missing.merge(temp_cmip,
                                      left_on=["DATEANDTIME", "lat_round", "lon_round", "LEV_M"],
                                      right_on=["year_bins", "lat_round", "lon_round", "lev_bins"],
                                      how="inner")

        df_compare[parameter] = df_compare[parameter] - df_compare[var_id]
        df_compare = df_compare[[parameter, "DATEANDTIME", "lat_round", "lon_round", "LEV_M"]].rename(columns={"lat_round": "LATITUDE", "lon_round": "LONGITUDE"})
        df_compares.append(df_compare)

    # Assemble parameters into one df
    temp = df_compares[0]
    for t in df_compares[1:]:
        temp = temp.merge(t, on=["DATEANDTIME", "LATITUDE", "LONGITUDE", "LEV_M"], how="outer")
    temp_group = temp.drop("DATEANDTIME", axis=1).groupby(["LATITUDE", "LONGITUDE", "LEV_M"])

    # Mean and std over time
    temp_mean = temp_group.mean().reset_index()
    temp_std = temp_group.std().reset_index()
    return {"mean": temp_mean, "std": temp_std}


In [ ]:
res_cmip = {}
res_cmip_scaled = {}

# Loop over models
for dir_name in test_dir_names:
    print(dir_name)

    model_out_dir = Path(dir_name)
    model_name = model_out_dir.parts[-1]
    exp_id = "" if model_out_dir.parts[-1] == model_name else model_out_dir.parts[-1]

    # Load data
    y_pred = pd.read_csv(model_out_dir / f"model{model_name}_y_pred.csv")
    y_true = pd.read_csv(model_out_dir / f"model{model_name}_y_true.csv")

    # Cleanup
    if "Unnamed: 0" in y_pred.columns:
        y_pred = y_pred.drop(columns="Unnamed: 0")

    # Inverse scaling
    y_pred_unscaled = y_pred.copy()
    scaler_dict = joblib.load(model_out_dir / "scalers.joblib")
    for k, v in scaler_dict.items():
        if k in y_pred_unscaled.columns:
            y_pred_unscaled[k] = v.inverse_transform(y_pred_unscaled[k].to_numpy().reshape(-1, 1))

    # Add coordinates
    y_pred[config.coordinates] = df[config.coordinates]
    df_rec = pd.DataFrame(y_pred)

    y_pred_unscaled[config.coordinates] = df[config.coordinates]
    df_rec_unscaled = pd.DataFrame(y_pred_unscaled)

    df_true = pd.DataFrame(y_true)
    df_true[config.coordinates] = df[config.coordinates]

    # Precompute rounded coords
    df_rec["lat_round"] = df_rec["LATITUDE"].round(1)
    df_rec["lon_round"] = df_rec["LONGITUDE"].round(1)

    df_rec_unscaled["lat_round"] = df_rec_unscaled["LATITUDE"].round(1)
    df_rec_unscaled["lon_round"] = df_rec_unscaled["LONGITUDE"].round(1)

    a = compute_cmip_diffs(df_rec=df_rec_unscaled, df_true=df_true, cmip_dict=cmip_dict, scaler_dict=None)
    b = compute_cmip_diffs(df_rec=df_rec, df_true=df_true, cmip_dict=cmip_dict, scaler_dict=scaler_dict)

    # Store
    res_cmip[model_name] = a
    res_cmip_scaled[model_name] = b

In [ ]:
res_to_plot = {"scaled": res_cmip_scaled, "unscaled": res_cmip}

for scaling, temp_res in res_to_plot.items():
    for parameter in config.parameters:
        print(parameter)

        # Compute norm for cmap
        df_all_param = []
        for model in [m.rstrip("_") for m in models_to_plot]:
            df_model = temp_res[model]["mean"]
            df_all_param.append(df_model[parameter].dropna())
        df_all_param = pd.concat(df_all_param)

        absmax = np.max(np.abs(df_all_param))
        mean_norm = mcolors.TwoSlopeNorm(vmin=-absmax, vcenter=0, vmax=absmax)

        plot_config = {
            "mean": {
                "cmap": "seismic",
                "norm": mean_norm,
            },
            "std": {
                "cmap": "viridis",
                "norm": None,
            }
        }

        plot_depth_maps(
            res_dict=temp_res,
            models_to_plot=[m.rstrip("_") for m in models_to_plot],
            imputer_name_map=imputer_name_map,
            depths=depths,
            parameter=parameter,
            plot_config=plot_config,
            nrows=nrows,
            ncols=ncols,
            mode_list=["mean", "std"],
            output_path="output/plots/test_evaluation/cmip_diff",
            prefix=f"cmip_diff_{scaling}",
            dpi=300
        )



In [ ]:
# ECDF plot (scaled CMIP diffs)
df_cmip_diff = []
for model_name, temp_res in res_cmip_scaled.items():
    temp = temp_res["mean"]
    temp["mean_diff"] = temp[config.parameters].abs().mean(axis=1)
    temp["Model"] = model_name
    df_cmip_diff.append(temp)

df_cmip_diff = pd.concat(df_cmip_diff)
df_cmip_diff["Model"] = df_cmip_diff["Model"].map(imputer_name_map)

# Plot
plt.figure(figsize=(6, 6))
hue_order = [imputer_name_map[o] for o in order]
ax = sns.ecdfplot(data=df_cmip_diff, x="mean_diff", hue="Model", hue_order=hue_order)
sns.move_legend(ax, loc="upper left")#, ncol=3)

ax.set_xlabel("|CMIP - pred|")
ax.set_ylabel("Proportion ≤ x")
ax.set_ylim(0, 1.01)

# Insert zoom plot
s = 45
axins = inset_axes(ax, width=f"{s}%", height=f"{0.05/0.05 * s}%", loc="lower right")
sns.ecdfplot(data=df_cmip_diff, x="mean_diff", hue="Model", hue_order=hue_order, ax=axins, legend=False)

# Zoom region
axins.set_xlim(0.12, 0.17)
axins.set_ylim(0.95, 1)
axins.set_title("Zoom", fontsize=9)
axins.set_xlabel("")
axins.set_ylabel("")

# Box around zoom region on original plot
rect = patches.Rectangle(xy=(0.12, 0.95), width=0.05, height=0.05, linewidth=1, edgecolor="gray", facecolor="none", linestyle="--", alpha=0.5)
ax.add_patch(rect)

# Get colors used by seaborn
medians = df_cmip_diff.groupby("Model", observed=True)["mean_diff"].median()
palette = sns.color_palette(n_colors=len(hue_order))
color_map = dict(zip(hue_order, palette))

for i, model in enumerate(models):
    x = medians[imputer_name_map[model]]
    ax.vlines(x=x, ymin=0, ymax=0.02, linewidth=1, color=color_map[imputer_name_map[model]])

plt.tight_layout()
plt.savefig(f"{config.output_dir_plots}/test_evaluation/cmip_diff/cmip_diff_ecdf.png", dpi=300)
plt.show()

# Ablation: Token representation

In [ ]:
# import json
# import pandas as pd
# import numpy as np
# import seaborn as sns
# from glob import glob
# from pathlib import Path
# import matplotlib.pyplot as plt
# from tuning_studies.modelconfig import ablation_study
#
# import config

baseline_cfg = ablation_study["exp0"]["config"]
model_name_map = {"space_time_attention": "Space-time attention", "mha": "MHA", "autoencoder": "Autoencoder", "encoder_decoder": "Autoencoder", "space_time_depth_attention": "Space-time-depth attention", "transformer_encoder_layer": "Transformer encoder", "transformer_encoder": "Transformer encoder", "time_space_attention": "Time-space attention"}

def get_attr(res, attr_name):
    if attr_name in res["hyps"]["model"]["cfg"].keys():
        return res["hyps"]["model"]["cfg"][attr_name]
    else:
        return getattr(baseline_cfg, attr_name)

def load_experiments(exp_list, attr_list, model_name="mastnet", file_path=None, woa=True, cmip=True):
    df_woa, df_cmip = [], []

    results = []
    for exp_id in exp_list:
        # Get (validation) metrics from relevant experiments
        file_path_temp = None
        if file_path is None:
            f1 = Path(f"D:/phd/output/tuning/{model_name}/architecture_ablation/exp{exp_id}/")
            f2 = Path(f"D:/phd/output/tuning/mastnet/architecture_ablation/exp{exp_id}/")

            if f1.is_dir():
                file_path_temp = f1
            elif f2.is_dir():
                file_path_temp = f2
            else:
                print(f"Experiment {exp_id} not found")
        else:
            file_path_temp = file_path

        res_file_path = file_path_temp / f"model{model_name}_splitfinal_trial0.json"
        res = json.load(open(res_file_path, "r"))

        metrics_key = "test_metrics" if res["model_framework"] == "pytorch" else "metrics_all"
        metrics = pd.DataFrame([res[metrics_key]["Global"]])

        metrics["exp_id"] = exp_id

        metrics["train_time"] = res["train_time"]
        metrics["pred_time"] = res["pred_time"]

        if not Path(res_file_path).is_file():
            print(f"No such file: {res_file_path}")
            return None, None, None

        # Get requested metrics
        for attr in attr_list:
            metrics[attr] = get_attr(res, attr)
        results.append(metrics)

        # WOA data (scaled)
        if woa:
            w = pd.read_csv(file_path_temp / f"{model_name}_woa_diff_summary_scaled.csv")
            w["exp_id"] = exp_id
            df_woa.append(w)

            metrics["WOA_diff"] = w["MAE"].mean()

        # CMIP data (scaled)
        if cmip:
            c = pd.read_csv(file_path_temp / f"{model_name}_cmip_diff_summary_scaled.csv")
            c["exp_id"] = exp_id
            df_cmip.append(c)

            metrics["CMIP_diff"] = c["MAE"].mean()

    results = pd.concat(results)
    if "attention_type" in results.columns:
        results["attention_type"] = results["attention_type"].map(model_name_map)

    df_woa = None if len(df_woa) == 0 else pd.concat(df_woa)
    df_cmip = None if len(df_cmip) == 0 else pd.concat(df_cmip)

    return results, df_woa, df_cmip

In [ ]:
# Sequence type ablation
res_mastnet1, df_woa, df_cmip = load_experiments(exp_list=[11], attr_list=["attention_type"]) # , model_name="mastnet")
res_mastnet1["Token type"] = "Multi-feature"

res_mastnet2, df_woa, df_cmip = load_experiments(exp_list=[47], attr_list=["attention_type"]) # , model_name="mastnet")
res_mastnet2["Token type"] = "Time point"

res_remasker, df_woa, df_cmip = load_experiments(exp_list=[""], attr_list=[], model_name="remasker", file_path=Path("output/tuning/remasker/ablation/"))
res_remasker["Token type"] = "Single-feature"

res = pd.concat([res_mastnet1, res_mastnet2, res_remasker])
res = res.sort_values("Token type")
res["Total time"] = res["train_time"] + res["pred_time"]

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, metric in zip(axes, ["RMSE", "CMIP_diff", "WOA_diff", "Total time"]):
    sns.barplot(data=res, x="Token type", y=metric, ax=ax)

    if metric == "CMIP_diff":
        ax.set_title(f"Log|CMIP - pred|")
        ax.set_yscale("log")
    elif metric == "WOA_diff":
        ax.set_title(f"|WOA - pred|")
    else:
        ax.set_title(metric)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(f"output/plots/ablation/tokens/token_type.png")
plt.show()

# Ablation: Attention type

In [ ]:
# Feature mixer ablation
exp_ids = [0, 3, 10, 11, 17, 18, 37, 72, 15, 27, 31, 245, 246, 247, 248, 249, 250, 251]
attr_list = ["attention_type", "feature_mixer_input", "feature_mixer"]
error_name = "RMSE"

results, df_woa, df_cmip = load_experiments(exp_ids, attr_list)
results = results.sort_values(error_name, ascending=False)
results["Feature mixer"] = np.select([~results["feature_mixer"], results["feature_mixer"] & (results["feature_mixer_input"] == "feat"), results["feature_mixer"] & (results["feature_mixer_input"] == "feat_mask")], ["None", "On features", "On features and masks"], default="")
results["Total time"] = results["train_time"] + results["pred_time"]

# Plot
for m in ["RMSE", "CMIP_diff", "WOA_diff", "Total time"]:
    sns.barplot(data=results, x="attention_type", hue="Feature mixer", y=m)
    plt.xlabel("")
    plt.xticks(rotation=90)

    if m == "CMIP_diff":
        plt.ylabel(f"Log|CMIP - pred|")
        plt.yscale("log")
    elif m == "WOA_diff":
        plt.ylabel(f"|WOA - pred|")
    else:
        plt.ylabel(m)

    # plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(f"output/plots/ablation/attention/feature_mixer_{m}.png")
    plt.show()


# Ablation: Masking strategies

In [ ]:
exp_map = {
    "autoencoder": {
        "masking": {
            "per_sample": ["exp182", "exp183", "exp184", "exp185", "exp186"],
            "transect": ["exp217", "exp223", "exp229"],
            "transect_random": ["exp218", "exp219", "exp220", "exp221", "exp222", "exp224", "exp225", "exp226", "exp227", "exp228", "exp230", "exp231", "exp232", "exp233", "exp234"],
            "sphere": ["exp187", "exp193", "exp199", "exp205", "exp211"],
            "sphere_random": ["exp188", "exp189", "exp190", "exp191", "exp192",
                                 "exp194", "exp195", "exp196", "exp197", "exp198",
                                 "exp200", "exp201", "exp202", "exp203", "exp204",
                                 "exp206", "exp207", "exp208", "exp209", "exp210",
                                 "exp212", "exp213", "exp214", "exp215", "exp216",
                                 ],
            "combined": ["exp244"],
            "per_feature": ["exp235", "exp236", "exp237", "exp238", "exp239", "exp240", "exp241", "exp242", "exp243"],
        }
    },
    "space_time_attention": {
        "masking": {
            "per_sample": ["exp73", "exp91", "exp92", "exp93", "exp94"],
            "transect": ["exp109", "exp114",  "exp174"],
            "transect_random": ["exp139", "exp140", "exp141", "exp142", "exp143", "exp144", "exp145", "exp146", "exp147", "exp148",  "exp175", "exp176", "exp177", "exp178", "exp179"],
            "sphere": ["exp104", "exp105", "exp106", "exp107", "exp108"],
            "sphere_random": ["exp149", "exp150", "exp151", "exp152", "exp153",  # mask_ratio=0.3
                                 "exp154", "exp155", "exp156", "exp157", "exp158",  # mask_ratio=0.4
                                 "exp159", "exp160", "exp161", "exp162", "exp163",  # mask_ratio=0.5
                                 "exp164", "exp165", "exp166", "exp167", "exp168",  # mask_ratio=0.7
                                 "exp169", "exp170", "exp171", "exp172", "exp173",  # mask_ratio=0.9
                                 ],
            "combined": ["exp180"],
            "per_feature": ["exp95", "exp96", "exp97", "exp98", "exp99", "exp100", "exp101", "exp102", "exp103"]
        }
    }
}

## Per-sample masking

In [ ]:
def assemble_df(exp_ids):
    per_sample = []
    for exp_id in exp_ids:
        masking_dir = ablation_base_dir / exp_id
        df_mask, a, b = load_experiment(dir_path=masking_dir, model_name="mastnet", woa=True, cmip=True)
        df_mask["effective_miss_ratio"] = extract_mean_miss_ratio(exp_id)

        df_mask["sphere_mask_radius"] = ablation_study[exp_id]["config"].sphere_mask_radius
        df_mask["transect_mask_width"] = ablation_study[exp_id]["config"].transect_mask_width
        df_mask["masking_strategies"] = str(ablation_study[exp_id]["config"].masking_strategies)

        per_sample.append(df_mask)

    df_per_sample = pd.concat(per_sample)
    df_per_sample["num_feat"] = df_per_sample["mask_ratio"].map(per_sample_num_map)
    df_per_sample["masking_strategy"] = "Random"

    return df_per_sample

In [ ]:
def plot_masking_rmse(df_in, save_as=None, dpi=300):
    plt.figure(figsize=(6, 4))
    sns.barplot(df_in, x="num_feat", y="RMSE", hue="Attention type")
    plt.xlabel("Number of randomly hidden features per sample")
    plt.tight_layout()

    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.show()

def plot_masking_woa_diff(df_in, save_as=None, dpi=300):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 4), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

    temp = df_in.sort_values("num_feat").copy()
    temp["num_feat"] = temp["num_feat"].astype(str)

    # Mean
    sns.barplot(data=temp, x="num_feat", y="WOA_diff", ax=ax1, hue="Attention type")
    ax1.set_yscale("log")
    ax1.set_ylabel("Log |WOA - pred|")

    # Std
    sns.scatterplot(data=temp, x="num_feat", y="WOA_std", ax=ax2, hue="Attention type", legend=False)
    ax2.set_yscale("log")
    ax2.set_ylabel("Log std")

    ax2.set_xlabel("Number of randomly hidden features per sample")

    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.show()

def plot_masking_cmip_diff(df_in, save_as=None, dpi=300):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 4), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

    temp = df_in.sort_values("num_feat").copy()
    temp["num_feat"] = temp["num_feat"].astype(str)

    # Mean
    sns.barplot(data=temp, x="num_feat", y="CMIP_diff", ax=ax1, hue="Attention type")
    ax1.set_yscale("log")
    ax1.set_ylabel("Log |CMIP -  pred|")

    # Std
    sns.scatterplot(data=temp, x="num_feat", y="CMIP_std", ax=ax2, hue="Attention type", legend=False)
    ax2.set_yscale("log")
    ax2.set_ylabel("Log std")

    ax2.set_xlabel("Number of randomly hidden features per sample")

    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.show()


In [ ]:
df_per_sample = []
for model_name, exp_dict in exp_map.items():
    exp_ids = exp_dict["masking"]["per_sample"]
    temp = assemble_df(exp_ids)
    temp["Attention type"] = attention_name_map[model_name]
    df_per_sample.append(temp)
df_per_sample = pd.concat(df_per_sample)

plot_masking_rmse(df_per_sample, save_as=f"output/plots/ablation/masking/masking_rmse.png", dpi=300)
plot_masking_cmip_diff(df_per_sample, save_as=f"output/plots/ablation/masking/masking_cmip_diff.png", dpi=300)
plot_masking_woa_diff(df_per_sample, save_as=f"output/plots/ablation/masking/masking_woa_diff.png", dpi=300)

## Transect masking

In [ ]:
def assemble_transect_df(exp_ids_transect, exp_ids_transect_random):
    transect = []
    for exp_id in exp_ids_transect + exp_ids_transect_random:
        masking_dir = ablation_base_dir / exp_id
        df_mask, a, b = load_experiment(dir_path=masking_dir, model_name="mastnet", woa=True, cmip=True)
        df_mask["effective_miss_ratio"] = extract_mean_miss_ratio(exp_id)

        if exp_id in exp_ids_transect:
            df_mask["num_feat"] = 0

        transect_width = ablation_study[exp_id]["config"].transect_mask_width
        df_mask["transect_mask_width"] = transect_width
        df_mask["sphere_mask_radius"] = ablation_study[exp_id]["config"].sphere_mask_radius

        df_mask["masking_strategies"] = str(ablation_study[exp_id]["config"].masking_strategies)
        df_mask["masking_strategy"] = f"Transect (w={transect_width})" if df_mask["transect_mask_orientation"].values[0] == 0 else f"Transect (w={transect_width}, aligned)"

        transect.append(df_mask)

    df_transect = pd.concat(transect)
    df_transect["num_feat"] = df_transect["mask_ratio"].map(per_sample_num_map)
    df_transect.loc[df_transect["exp_id"].isin(exp_ids_transect), "num_feat"] = 0

    return df_transect

In [ ]:
df_transect = []

for model_name, exp_dict in exp_map.items():
    exp_ids_transect = exp_dict["masking"]["transect"]
    exp_ids_transect_random = exp_dict["masking"]["transect_random"]
    temp = assemble_transect_df(exp_ids_transect, exp_ids_transect_random)
    temp["Attention type"] = attention_name_map[model_name]
    df_transect.append(temp)

df_transect = pd.concat(df_transect)

## Spherical masking

In [ ]:
def assemble_sphere_df(exp_ids_sphere, exp_ids_sphere_random):
    sphere = []
    for exp_id in exp_ids_sphere + exp_ids_sphere_random:
        masking_dir = ablation_base_dir / exp_id
        df_mask, a, b = load_experiment(dir_path=masking_dir, model_name="mastnet", woa=True, cmip=True)
        df_mask["effective_miss_ratio"] = extract_mean_miss_ratio(exp_id)

        df_mask["transect_mask_width"] = ablation_study[exp_id]["config"].transect_mask_width
        df_mask["sphere_mask_radius"] = ablation_study[exp_id]["config"].sphere_mask_radius
        df_mask["masking_strategies"] = str(ablation_study[exp_id]["config"].masking_strategies)
        df_mask["masking_strategy"] = f"Sphere (r=" + str(df_mask["sphere_mask_radius"].values[0]) + ")"
        sphere.append(df_mask)

    df_spherical = pd.concat(sphere)
    df_spherical["num_feat"] = df_spherical["mask_ratio"].map(per_sample_num_map)
    df_spherical.loc[df_spherical["exp_id"].isin(exp_ids_sphere), "num_feat"] = 0

    return df_spherical

In [ ]:
df_sphere = []

for model_name, exp_dict in exp_map.items():
    exp_ids_sphere = exp_dict["masking"]["sphere"]
    exp_ids_sphere_random = exp_dict["masking"]["sphere_random"]
    temp = assemble_sphere_df(exp_ids_sphere, exp_ids_sphere_random)
    temp["Attention type"] = attention_name_map[model_name]
    df_sphere.append(temp)

df_sphere = pd.concat(df_sphere)

## Combined

In [ ]:
def assemble_combined_df(exp_ids):
    combined = []
    for exp_id in exp_ids_combined:
        masking_dir = ablation_base_dir / exp_id
        df_mask, a, b = load_experiment(dir_path=masking_dir, model_name="mastnet", woa=True, cmip=True)
        df_mask["effective_miss_ratio"] = extract_mean_miss_ratio(exp_id)

        if exp_id in ["exp244"]:
            df_mask["num_feat"] = 1

        df_mask["transect_mask_width"] = ablation_study[exp_id]["config"].transect_mask_width
        df_mask["sphere_mask_radius"] = ablation_study[exp_id]["config"].sphere_mask_radius
        df_mask["masking_strategies"] = str(ablation_study[exp_id]["config"].masking_strategies)

        df_mask["masking_strategy"] = "Combined"
        combined.append(df_mask)

    df_combined = pd.concat(combined)
    df_combined["num_feat"] = df_combined["mask_ratio"].map(per_sample_num_map)
    df_combined.loc[df_combined["exp_id"].isin(exp_ids_combined), "num_feat"] = 0

    return df_combined

def plot_masking_heatmap(df, error_name="RMSE", save_as=None, dpi=150):

    # Pivot for RMSE
    error_pivot = df.pivot_table(
        index="masking_strategy",
        columns="num_feat",
        values=error_name,
        aggfunc="mean"
    ).sort_index()

    # Pivot effective miss ratio
    miss_pivot = None
    if "effective_miss_ratio" in df.columns and df["effective_miss_ratio"].notna().any():
        miss_pivot = df.pivot_table(
            index="masking_strategy",
            columns="num_feat",
            values="effective_miss_ratio",
            aggfunc="mean"
        ).sort_index()

    # Plot
    fig, axes = plt.subplots(1, 2 if miss_pivot is not None else 1, figsize=(14, 5))
    ax1, ax2 = axes

    # Error heatmap
    sns.heatmap(error_pivot, annot=True, fmt=".3f", cmap="magma", ax=ax1)
    ax1.set_title(f"{error_name}")
    ax1.set_xlabel("Num. features masked")
    ax1.set_ylabel("Masking strategy")

    # Effective miss ratio heatmap
    sns.heatmap(miss_pivot, annot=True, fmt=".4f", cmap="magma", ax=ax2)
    ax2.set_title("Effective Mask Ratio")
    ax2.set_xlabel("Num. features masked")
    ax2.set_ylabel("")

    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.show()

In [ ]:
df_combined = []

for model_name, exp_dict in exp_map.items():
    exp_ids_combined = exp_dict["masking"]["combined"]
    temp = assemble_combined_df(exp_ids_combined)
    temp["Attention type"] = attention_name_map[model_name]
    df_combined.append(temp)

df_combined = pd.concat(df_combined)

## Heatmaps

In [ ]:
df_all = pd.concat([df_per_sample, df_transect, df_sphere, df_combined], ignore_index=True)

for model_name in df_all["Attention type"].unique():
    print(model_name)
    for error_name in ["RMSE", "CMIP_diff", "WOA_diff"]:
        temp = df_all[df_all["Attention type"] == model_name]
        plot_masking_heatmap(temp, error_name=error_name, save_as=f"output/plots/ablation/masking/masking_heatmap_{model_name}_{error_name}.png")

In [ ]:
# Error correlations
for model_name in df_all["Attention type"].unique():
    print(model_name)
    temp = df_all[df_all["Attention type"] == model_name]
    temp_corr = temp[["RMSE", "WOA_diff", "CMIP_diff", "effective_miss_ratio"]].corr()
    print(temp_corr)
    print()

## Per-feature masking

In [ ]:
def assemble_per_feature_df(exp_ids):
    per_feature = []
    for exp_id in exp_ids:
        masking_dir = ablation_base_dir / exp_id
        df_mask, a, b = load_experiment(dir_path=masking_dir, model_name="mastnet", woa=True, cmip=True)
        df_mask["effective_miss_ratio"] = extract_mean_miss_ratio(exp_id)
        per_feature.append(df_mask)

    df_per_feature = pd.concat(per_feature)
    return df_per_feature

def plot_masking_per_feature_error(df_in, x="mask_ratio", y="RMSE", y_label=None, y_scale=None, save_as=None, dpi=300):
    plt.figure(figsize=(6, 4))
    sns.barplot(df_in, x=x, y=y, hue="Attention type")
    plt.xlabel("Mask ratio")
    if y_label:
        plt.ylabel(y_label)
    if y_scale:
        plt.yscale(y_scale)

    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.show()

In [ ]:
df_per_feature = []

for model_name, exp_dict in exp_map.items():
    exp_ids = exp_dict["masking"]["per_feature"]
    temp = assemble_per_feature_df(exp_ids)
    temp["Attention type"] = attention_name_map[model_name]
    df_per_feature.append(temp)

df_per_feature = pd.concat(df_per_feature)

plot_masking_per_feature_error(df_per_feature, x="mask_ratio", y="RMSE", save_as="output/plots/ablation/masking/masking_per_feature_rmse.png", dpi=300)
plot_masking_per_feature_error(df_per_feature, x="mask_ratio", y="CMIP_diff", y_label="Log|CMIP - pred|", y_scale="log", save_as="output/plots/ablation/masking/masking_per_feature_CMIP_diff.png", dpi=300)
plot_masking_per_feature_error(df_per_feature, x="mask_ratio", y="WOA_diff", y_label="|WOA - pred|", save_as="output/plots/ablation/masking/masking_per_feature_WOA_diff.png", dpi=300)

In [ ]:
# Error correlations
for model_name in df_per_feature["Attention type"].unique():
    print(model_name)
    temp = df_per_feature[df_per_feature["Attention type"] == model_name]
    temp_corr = temp[["RMSE", "WOA_diff", "CMIP_diff", "effective_miss_ratio"]].corr()
    print(temp_corr)
    print()